# 03 — Demo in Colab (Gradio, same code as the Streamlit app)

Use this when the venue has no hosted app, or as a backup to Streamlit Cloud.
It runs `src/inference.py` directly, so a bug fixed in the app is fixed here too.

**You need:** `models/detect/best.pt` (and `models/classify/best.pt` for Dive Deeper) uploaded to Colab. Without them, the demo stops with the training step — it will not pretend a COCO model detects corrosion.

In [ ]:
# Corrosion Vision — Colab setup
# 1. Runtime -> Change runtime type -> GPU (T4 is enough for nano models).
# 2. Upload your GitHub repo (or this folder) to Colab, and cd into it.
# 3. Upload kaggle.json (Kaggle -> Settings -> API) to Colab first.
import os, shutil
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
if os.path.exists("kaggle.json"):
    shutil.copy("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    print("kaggle.json installed (never commit it).")
else:
    print("kaggle.json not found in this folder — upload it, then re-run this cell.")
!pip install -q -r requirements.txt
!python -c "import ultralytics; print('ultralytics', ultralytics.__version__)"
!pip install -q gradio
!nvidia-smi | head -12


In [ ]:
import gradio as gr, cv2, numpy as np, sys
sys.path.insert(0, ".")
from src.inference import load_yolo, predict

detect = load_yolo("models/detect/best.pt", "detection (notebook 01)")
print("Detector loaded. Upload a validation image below.")

def demo(image, dive_deeper, conf):
    if image is None:
        return "Upload an image first.", None
    bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    clf = None
    if dive_deeper:
        clf = load_yolo("models/classify/best.pt", "classification (notebook 02)")
    res = predict(bgr, detect, clf, dive_deeper=dive_deeper, conf=conf)
    ann = cv2.cvtColor(res.annotated_bgr, cv2.COLOR_BGR2RGB)
    lines = [f"Verdict: {res.verdict}  |  Boxes: {len(res.detections)}"]
    if res.severity:
        s = res.severity
        lines += [f"Severity %: {s['severity_pct']}%  |  ASTM D610 grade: {s['astm_d610_grade']}  |  "
                  f"Box-area % (upper bound): {s['box_area_pct']}%  |  Distribution: {s['distribution']}"]
        lines.append(s["recommendation"])
    if res.type_prediction:
        lines += [f"Type: {label} ({prob:.1%})" for label, prob in res.type_prediction]
    return "\n".join(lines), ann

gr.Interface(
    fn=demo,
    inputs=[gr.Image(label="Steel-surface photo"),
            gr.Checkbox(label="Dive Deeper — type + severity %", value=False),
            gr.Slider(0.05, 0.95, 0.25, step=0.05, label="Confidence threshold")],
    outputs=[gr.Textbox(label="Result"), gr.Image(label="Annotated image")],
    title="Corrosion Vision — Colab demo",
    description="Visual screening only, not a certified inspection. See docs/STANDARDS.md.",
).launch(share=True)
